<a href="https://colab.research.google.com/github/bercyx27/PYTHON-BIOLOGY-AND-CHEMISTRY/blob/main/Project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project 1
Procedural Programming

In [7]:
import requests
import datetime
from IPython.display import HTML, display

# --- 1. Data Fetching Function ---
def fetch_weather_data(lat, lon):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat, "longitude": lon,
        "current": ["temperature_2m", "weather_code"],
        "hourly": ["temperature_2m", "precipitation_probability"],
        "daily": ["weather_code", "temperature_2m_max", "temperature_2m_min", "precipitation_probability_max"],
        "timezone": "auto", "forecast_days": 5
    }

    try:
        response = requests.get(url, params=params, timeout=5)
        if response.status_code == 200:
            return response.json()
    except:
        pass

    # Fallback data if offline
    return {
        "current": {"temperature_2m": 26.0, "weather_code": 0},
        "hourly": {"time": ["23:00", "00:00", "01:00", "02:00", "03:00"], "temperature_2m": [25, 24, 23, 22, 22], "precipitation_probability": [0, 1, 3, 5, 7]},
        "daily": {"time": ["Sat", "Sun", "Mon", "Tue", "Wed"], "weather_code": [0, 0, 1, 2, 2], "temperature_2m_max": [30, 30, 30, 30, 30], "temperature_2m_min": [18, 18, 18, 18, 10], "precipitation_probability_max": [10, 15, 20, 15, 10]}
    }

# --- 2. Utility Function ---
def get_svg(icon_type):
    if icon_type == "moon": return '''<svg width="28" height="28" viewBox="0 0 24 24" fill="none"><path d="M21 12.75C21 17.8576 16.8576 22 11.75 22C8.3073 22 5.31969 20.1085 3.7915 17.2915C6.0123 17.8488 8.4414 17.3486 10.2319 15.5581C12.0224 13.7676 12.5226 11.3385 11.9653 9.1177C14.7823 7.58951 16.6738 4.6019 16.6738 1.16C21.7814 1.16 25.9238 5.3024 25.9238 10.41" fill="#f1c40f"/></svg>'''
    elif icon_type == "sun": return '''<svg width="28" height="28" viewBox="0 0 24 24" fill="none"><circle cx="12" cy="12" r="5" fill="#f39c12"/><path d="M12 1v3M12 20v3M4.22 4.22l2.12 2.12M17.66 17.66l2.12 2.12M1 12h3M20 12h3M4.22 19.78l2.12-2.12M17.66 6.34l2.12-2.12" stroke="#f39c12" stroke-width="2" stroke-linecap="round"/></svg>'''
    return '''<svg width="28" height="28" viewBox="0 0 24 24" fill="none"><path d="M19.384 13.5a4.5 4.5 0 0 0-8.634-1.5A5.5 5.5 0 1 0 12 23h7.384a3.5 3.5 0 0 0 0-7z" fill="#ecf0f1"/></svg>'''

# --- 3. UI Rendering Function ---
def render_ui(data, city_name):
    # Process Hourly
    hourly_html = "".join([f'<div style="text-align:center; flex:1"><div style="font-size:13px; opacity:0.8">{t[-5:] if "T" in t else t}</div>{get_svg("moon")}<div style="font-weight:bold; margin:5px 0">{int(temp)}°C</div><div style="font-size:11px; color:#74b9ff">💧 {p}%</div></div>' for t, temp, p in zip(data['hourly']['time'][:5], data['hourly']['temperature_2m'][:5], data['hourly']['precipitation_probability'][:5])])

    # Process Daily
    daily_html = "".join([f'<div style="display:flex; align-items:center; margin-bottom:12px"><div style="width:45px; font-weight:500">{datetime.datetime.strptime(t, "%Y-%m-%d").strftime("%a") if "-" in t else t}</div><div style="width:35px">{get_svg("sun" if c < 2 else "cloud")}</div><div style="flex:1; display:flex; align-items:center; gap:8px"><span style="width:30px; font-size:12px">{int(min_t)}°</span><div style="flex:1; height:6px; background:rgba(255,255,255,0.1); border-radius:3px; position:relative"><div style="position:absolute; height:100%; left:{((min_t-10)/25)*100}%; width:{((max_t-min_t)/25)*100}%; background:linear-gradient(90deg, #74b9ff, #f39c12); border-radius:3px"></div></div><span style="width:30px; font-size:12px; text-align:right">{int(max_t)}°</span></div><div style="width:45px; text-align:right; font-size:12px; color:#74b9ff">💧 {p}%</div></div>' for t, max_t, min_t, p, c in zip(data['daily']['time'], data['daily']['temperature_2m_max'], data['daily']['temperature_2m_min'], data['daily']['precipitation_probability_max'], data['daily']['weather_code'])])

    html = f'''
    <div style="max-width: 440px; margin: 20px auto; background: #1a2536; border-radius: 30px; padding: 24px; color: white; font-family: sans-serif; box-shadow: 0 10px 30px rgba(0,0,0,0.5);">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:15px">
            <h1 style="margin:0; font-size:28px">{city_name}</h1>
            <div style="background:rgba(255,255,255,0.1); border-radius:50%; padding:5px 10px; cursor:pointer">•••</div>
        </div>
        <div style="display:flex; justify-content:space-between; margin-bottom:20px">
            <h2 style="margin:0; font-size:82px; font-weight:200">{int(data['current']['temperature_2m'])}°C</h2>
            {get_svg("cloud")}
        </div>
        <div style="background:rgba(230,126,34,0.2); border-left:4px solid #e67e22; padding:10px; border-radius:8px; margin-bottom:20px; color:#f39c12; font-weight:bold; font-size:13px">⚠️ Extreme Heat Warning - See Details</div>
        <div style="background:rgba(255,255,255,0.06); padding:15px; border-radius:20px; margin-bottom:15px">
            <h3 style="margin:0 0 15px 0; font-size:13px; color:rgba(255,255,255,0.6)">TODAY\'S HOURLY</h3>
            <div style="display:flex; justify-content:space-between">{hourly_html}</div>
        </div>
        <div style="background:rgba(255,255,255,0.06); padding:15px; border-radius:20px; margin-bottom:15px">
            <h3 style="margin:0 0 15px 0; font-size:13px; color:rgba(255,255,255,0.6)">5-DAY FULL FORECAST</h3>
            {daily_html}
        </div>
    </div>'''
    display(HTML(html))

# --- 4. Main Execution (The Procedure) ---
city = "Stuttgart"
weather_data = fetch_weather_data(48.78, 9.18) # Pass coords
render_ui(weather_data, city)                  # Pass data and city to renderer

# New Section